[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Impact-Map/opm_ehri_data/blob/va-section505-vs-opm-analysis/va_section505_vs_opm.ipynb)

*On Colab: uncomment the `!pip install` line in the setup cell.*

# VA §505 vs. OPM/EHRI — do they agree?

**The puzzle:** OPM's VA *headcount* sits a bit *above* the §505 onboard count (expected — §505 excludes some staff), but OPM's *accessions/separations* looked *below* §505 — the opposite direction.

**The answer, in one line:** that flow result was an apples-to-oranges artifact. Compare the **same fiscal quarter** on **net change**, and OPM and §505 agree to ~3%. Gross accessions/separations aren't comparable across the two systems — and the entire difference is **VHA**.

Three checks below: (1) headcount, (2) flows the right way, (3) where the gap lives. Methodology notes at the end. All figures run live against `impactproject/opm-ehri-data`; §505 numbers are verified against the published `Section-505-FY26-Q2.xlsx`.

In [1]:
# !pip install -q huggingface_hub pandas pyarrow   # uncomment on Colab
import re, pandas as pd
from huggingface_hub import hf_hub_download, list_repo_files

REPO = "impactproject/opm-ehri-data"
_FILES = list_repo_files(REPO, repo_type="dataset")

def load(kind, ym, columns=None):
    """Load the latest version of an OPM file, VA rows only."""
    v = max(int(m.group(1)) for f in _FILES
            if (m := re.match(rf"{kind}/{kind}_{ym}_v(\d+)\.parquet", f)))
    df = pd.read_parquet(hf_hub_download(REPO, f"{kind}/{kind}_{ym}_v{v}.parquet",
                                         repo_type="dataset"), columns=columns)
    return df[df["agency_code"] == "VA"].copy()

def comparable(d):
    """Drop the §505 exclusions we can tag in OPM: OIG, intermittent, student-trainees."""
    return d[(d.agency_subelement != "INSPECTOR GENERAL")
             & (d.work_schedule != "INTERMITTENT")
             & (~d.occupational_series.str.contains("STUDENT TRAINEE", na=False))]

def admin_of(s):
    return {"VETERANS HEALTH ADMINISTRATION": "VHA",
            "VETERANS BENEFITS ADMINISTRATION": "VBA",
            "NATIONAL CEMETERY ADMINISTRATION": "NCA"}.get(s, "Staff Offices")

Q2 = ["202601", "202602", "202603"]   # FY2026 Q2 = Jan+Feb+Mar 2026
FLOW_COLS = ["agency_code", "agency_subelement", "work_schedule",
             "occupational_series", "count"]

# §505 FY2026 Q2, verified verbatim from Section-505-FY26-Q2.xlsx
S505 = {"onboard": 434636, "acc": 6637, "sep": 8058}
S505_ACC = {"VHA": 6534, "VBA": 11, "NCA": 52, "Staff Offices": 40}
S505_SEP = {"VHA": 7307, "VBA": 529, "NCA": 57, "Staff Offices": 165}

/Users/abigailhaddad/Documents/repos/pull_usaspending/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Headcount: OPM is higher, and that's expected

§505 excludes OIG, Veterans Canteen Service, intermittent staff, and (for VHA) residents/interns/fellows/students/trainees, plus non-pay-status employees. The taggable ones explain about half the ~12k gap; the rest is non-pay-status + VHA's paid trainees + ~1 month of timing. (Canteen and residents/interns/fellows aren't in OPM *at all* — VCS is non-appropriated, residents are non-salaried.)

In [2]:
emp = load("employment", "202604",
           columns=["agency_code", "agency_subelement", "work_schedule", "occupational_series"])
tot    = len(emp)
oig    = int((emp.agency_subelement == "INSPECTOR GENERAL").sum())
interm = int(((emp.work_schedule == "INTERMITTENT") & (emp.agency_subelement != "INSPECTOR GENERAL")).sum())

print(f"OPM VA (April 2026):                 {tot:,}")
print(f"  - OIG:                             {oig:,}   (in OPM, excluded by §505)")
print(f"  - intermittent:                    {interm:,}   (in OPM, excluded by §505)")
print(f"  - Canteen / residents / interns:   0       (not in OPM at all)")
print(f"  = after tagged exclusions:         {tot - oig - interm:,}")
print(f"§505 onboard (Mar 31, 2026):         {S505['onboard']:,}")
print(f"  residual gap:                      {tot - oig - interm - S505['onboard']:,}   "
      f"(non-pay-status + VHA paid trainees + timing)")

OPM VA (April 2026):                 446,735
  - OIG:                             956   (in OPM, excluded by §505)
  - intermittent:                    4,517   (in OPM, excluded by §505)
  - Canteen / residents / interns:   0       (not in OPM at all)
  = after tagged exclusions:         441,262
§505 onboard (Mar 31, 2026):         434,636
  residual gap:                      6,626   (non-pay-status + VHA paid trainees + timing)


## 2. Flows: compare the same quarter, on net change

§505 reports accessions/separations as **quarterly** totals; OPM's are **monthly** — so sum OPM to the fiscal quarter (and drop the same excluded groups). Do that and **net change agrees to ~3%**. *Gross* flows run ~30% higher in OPM on both sides — symmetrically, so net is preserved. (Don't compare a single OPM month, and beware: recent OPM months are revised, e.g. Feb 2026 is still light.)

In [3]:
def q2(kind):
    raw = comp = 0
    for ym in Q2:
        d = load(kind, ym, columns=FLOW_COLS)
        d["count"] = pd.to_numeric(d["count"])
        raw  += d["count"].sum()
        comp += comparable(d)["count"].sum()
    return int(raw), int(comp)

ar, ac = q2("accessions")
sr, sc = q2("separations")
print(pd.DataFrame([
    ["Accessions",  ar,      ac,      S505["acc"]],
    ["Separations", sr,      sc,      S505["sep"]],
    ["Net",         ar - sr, ac - sc, S505["acc"] - S505["sep"]],
], columns=["FY26 Q2", "OPM raw", "OPM 505-comp", "§505"]).to_string(index=False))

    FY26 Q2  OPM raw  OPM 505-comp  §505
 Accessions     9398          9001  6637
Separations    10654         10376  8058
        Net    -1256         -1375 -1421


## 3. The entire gap is VHA

Split the flows by administration: **VBA, NCA, and Staff Offices match within a few dozen actions**; ~95% of the gross difference is **VHA** (and the headcount gap is too). VHA is the one place with Title 38 hiring and §505's extra "VHA only" exclusions, and it runs through a different HR pipeline than OPM's feed. So: VBA/NCA/Staff Offices are safe to compare on gross flows; **VHA is not** — but it all washes out in net change.

In [4]:
opm = {}
for kind in ["accessions", "separations"]:
    g = pd.concat([comparable(load(kind, ym, columns=FLOW_COLS)) for ym in Q2])
    g["count"] = pd.to_numeric(g["count"])
    g["admin"] = g.agency_subelement.map(admin_of)
    opm[kind] = g.groupby("admin")["count"].sum()

print(pd.DataFrame([
    [a, int(opm["accessions"].get(a, 0)), S505_ACC[a],
        int(opm["separations"].get(a, 0)), S505_SEP[a]]
    for a in ["VHA", "VBA", "NCA", "Staff Offices"]
], columns=["admin", "OPM acc", "§505 acc", "OPM sep", "§505 sep"]).to_string(index=False))

        admin  OPM acc  §505 acc  OPM sep  §505 sep
          VHA     8818      6534     9474      7307
          VBA       61        11      594       529
          NCA       68        52       99        57
Staff Offices       54        40      209       165


## Methodology & bottom line

**The two sources define these almost identically** — both are *personnel actions*, per-agency, active-pay-status, with the same exclusions. From the §505 FY26 Q2 report, verbatim:

> *"Accessions are personnel actions that result in an employee's addition to VA (i.e., transfers-in from another agency and new hires to the Federal Government)... Separations are personnel actions resulting in the loss of an employee from VA (i.e., transfers-out, resignations, retirements, terminations or removals, death, and other separations)."*

So the ~30% gross gap is **not** a definitional difference. It's a cross-system artifact: §505 is built from VA's **HRSmart / USA Staffing**, OPM from the **EHRI** feed, and they count/time VHA's Title 38 actions differently. Tellingly, §505's *own* report notes its gross flows don't reconcile to its own onboard change (net −1,421 vs onboard change −2,196, "additional personnel actions... executed in HRSmart after the data is extracted"). If gross doesn't tie to stock inside one system, it won't match across two.

**Bottom line — how to compare OPM and §505:**
1. Match the **fiscal quarter** (sum OPM months); never put one OPM month against a §505 quarter.
2. Compare **net change**, not gross accessions/separations.
3. Apply the same population exclusions. Outside VHA you can compare gross too; inside VHA, don't.

*Data: `impactproject/opm-ehri-data` (Hugging Face). §505: https://department.va.gov/employees/va-mission-act-section-505-data/*